# Implementación del Proyector MoE Ligero (LightSignMoEProjector)

Este notebook tiene como objetivo enseñarte cómo fusionar los tensores provenientes de **MediaPipe** (Rostro, Manos, Pose) utilizando una arquitectura **Mixture-of-Experts (MoE)** con **Soft-Gating** y consciencia temporal.

In [1]:
import os
import sys
import torch
import torch.nn as nn
import torch.nn.functional as F

# Añadir el directorio raíz al path para poder importar módulos de src/
ROOT_DIR = os.path.abspath('..')
if ROOT_DIR not in sys.path:
    sys.path.append(ROOT_DIR)

from src.models.landmark_encoder import CausalConv1dBlock, SlidingPerceiverResampler

print(f"[*] PyTorch version: {torch.__version__}")
print("[*] Módulos de src.models.landmark_encoder importados correctamente.")

[*] PyTorch version: 2.12.0+cu130
[*] Módulos de src.models.landmark_encoder importados correctamente.


## 1. Generación de Datos Simulados (MediaPipe Mock)

MediaPipe procesa video frame por frame. Para cada frame, nos entrega coordenadas (X, Y, Z) y a menudo un *visibility score* o confianza. Vamos a simular un lote (batch) de videos.

In [2]:
def generate_dummy_mediapipe_data(batch_size=2, seq_len=30):
    """
    Simula la salida de MediaPipe. 
    MediaPipe nos da: Manos (21 pts x 2), Rostro (Subset de 140 pts), Pose (33 pts).
    Para simplificar, simularemos ya los features aplanados y concatenados con sus scores de visibilidad promediados.
    """
    # Dimensiones (puntos * 3 coordenadas)
    face_dim = 140 * 3    # 420
    hands_dim = (21+21) * 3 # 126
    pose_dim = 33 * 3     # 99
    
    # Tensores espaciales (Coordenadas normalizadas)
    face_feats = torch.randn(batch_size, seq_len, face_dim)
    hands_feats = torch.randn(batch_size, seq_len, hands_dim)
    pose_feats = torch.randn(batch_size, seq_len, pose_dim)
    
    # Scores de visibilidad (MediaPipe Confidence). Entre 0 y 1.
    # Simulamos que a veces las manos no se ven (score bajo)
    face_vis = torch.rand(batch_size, seq_len, 1) * 0.5 + 0.5   # Siempre bastante visible
    pose_vis = torch.rand(batch_size, seq_len, 1) * 0.4 + 0.6   # Mayormente visible
    hands_vis = torch.rand(batch_size, seq_len, 1)              # Variabilidad alta (oclusiones frecuentes)
    
    return (face_feats, hands_feats, pose_feats), (face_vis, hands_vis, pose_vis)

features, visibilities = generate_dummy_mediapipe_data()
face_f, hands_f, pose_f = features
face_v, hands_v, pose_v = visibilities

print(f"Face Shape: {face_f.shape} | Vis: {face_v.shape}")
print(f"Hands Shape: {hands_f.shape} | Vis: {hands_v.shape}")
print(f"Pose Shape: {pose_f.shape} | Vis: {pose_v.shape}")

Face Shape: torch.Size([2, 30, 420]) | Vis: torch.Size([2, 30, 1])
Hands Shape: torch.Size([2, 30, 126]) | Vis: torch.Size([2, 30, 1])
Pose Shape: torch.Size([2, 30, 99]) | Vis: torch.Size([2, 30, 1])


## 2. Construcción del Enrutador (Router)
El enrutador observará todos los features y las visibilidades para decidir el peso de cada experto en cada frame.

In [3]:
class VisibilityAwareRouter(nn.Module):
    def __init__(self, face_dim=420, hands_dim=126, pose_dim=99):
        super().__init__()
        # +3 por los 3 valores de visibilidad
        total_dim = face_dim + hands_dim + pose_dim + 3 
        self.net = nn.Sequential(
            nn.Linear(total_dim, 128),
            nn.LayerNorm(128),
            nn.Tanh(),
            nn.Linear(128, 3) # Un peso para: Face, Hands, Pose
        )
        
    def forward(self, face_f, hands_f, pose_f, face_v, hands_v, pose_v):
        # Concatenar todo a lo largo de la última dimensión
        x = torch.cat([face_f, hands_f, pose_f, face_v, hands_v, pose_v], dim=-1)
        # Calcular logits y luego Softmax para obtener pesos que sumen 1
        logits = self.net(x)
        weights = F.softmax(logits, dim=-1)
        return weights # [B, T, 3]

router = VisibilityAwareRouter()
weights = router(face_f, hands_f, pose_f, face_v, hands_v, pose_v)
print(f"Pesos de Enrutamiento (Gating): {weights.shape}")
print(f"Pesos del Frame 0, Batch 0: {weights[0, 0, :].detach().numpy()}")

Pesos de Enrutamiento (Gating): torch.Size([2, 30, 3])
Pesos del Frame 0, Batch 0: [0.4733699  0.29530993 0.23132017]


## 3. Construcción del Proyector MoE Completo
Ahora ensamblaremos los expertos. Les añadiremos una proyección inicial, y usaremos tu clase `CausalConv1dBlock` para que cada experto entienda el movimiento antes de que sus salidas sean fusionadas.

In [4]:
class ExpertStream(nn.Module):
    """
    Cada experto procesa un flujo de datos (Rostro, Manos o Pose). 
    Primero proyecta a un espacio latente común, luego aplica consciencia temporal.
    """
    def __init__(self, in_dim, proj_dim=512):
        super().__init__()
        # Proyección Espacial Independiente
        self.spatial_proj = nn.Sequential(
            nn.Linear(in_dim, proj_dim),
            nn.LayerNorm(proj_dim),
            nn.GELU()
        )
        # Integración Temporal (usando la clase de tu landmark_encoder.py)
        self.temporal_conv = CausalConv1dBlock(d_model=proj_dim, kernel_size=3)
        
    def forward(self, x):
        # x: [B, T, in_dim]
        spatial_feat = self.spatial_proj(x)  # [B, T, proj_dim]
        temp_feat = self.temporal_conv(spatial_feat) # [B, T, proj_dim]
        return temp_feat

class LightSignMoEProjector(nn.Module):
    def __init__(self, face_dim=420, hands_dim=126, pose_dim=99, proj_dim=512, d_llm=2560):
        super().__init__()
        
        # 1. Pool de Expertos
        self.expert_face = ExpertStream(face_dim, proj_dim)
        self.expert_hands = ExpertStream(hands_dim, proj_dim)
        self.expert_pose = ExpertStream(pose_dim, proj_dim)
        
        # 2. Router Dinámico
        self.router = VisibilityAwareRouter(face_dim, hands_dim, pose_dim)
        
        # 3. Compresión Temporal y Proyección al LLM
        # Utilizamos el Resampler que ya creaste para pasar de, ej. 30 frames a 5-10 tokens densos.
        self.resampler = SlidingPerceiverResampler(d_model=proj_dim, d_llm=d_llm, window_size=4, stride=2)
        
    def forward(self, features, visibilities):
        face_f, hands_f, pose_f = features
        face_v, hands_v, pose_v = visibilities
        
        # Obtener pesos del router [B, T, 3]
        g_weights = self.router(face_f, hands_f, pose_f, face_v, hands_v, pose_v)
        
        # Procesar con Expertos [B, T, proj_dim]
        out_face = self.expert_face(face_f)
        out_hands = self.expert_hands(hands_f)
        out_pose = self.expert_pose(pose_f)
        
        # Fusión Ponderada (Multiplicar cada salida por su peso correspondiente y sumar)
        # .unsqueeze(-1) para que broadcasting funcione de [B, T] a [B, T, proj_dim]
        fused_embeddings = (
            g_weights[:, :, 0].unsqueeze(-1) * out_face +
            g_weights[:, :, 1].unsqueeze(-1) * out_hands +
            g_weights[:, :, 2].unsqueeze(-1) * out_pose
        )
        
        # Compresión Temporal y Proyección al LLM [B, T_condensado, d_llm]
        sign_tokens = self.resampler(fused_embeddings)
        
        return sign_tokens

# Instanciar el proyector completo
moe_projector = LightSignMoEProjector()

# Forward pass
llm_tokens = moe_projector(features, visibilities)

print(f"[*] Éxito. Dimensiones de entrada original: Secuencia de {face_f.shape[1]} frames.")
print(f"[*] Dimensiones de salida al LLM: {llm_tokens.shape}")
print("    -> Esto significa que hemos condensado la secuencia temporal a menos tokens, ")
print("    -> y los hemos proyectado a la dimensión requerida (2560).")

[*] Éxito. Dimensiones de entrada original: Secuencia de 30 frames.
[*] Dimensiones de salida al LLM: torch.Size([2, 14, 2560])
    -> Esto significa que hemos condensado la secuencia temporal a menos tokens, 
    -> y los hemos proyectado a la dimensión requerida (2560).
